In [3]:
import pandas as pd
df =pd.read_parquet(r'C:\Users\User\Documents\HFT\BACK_TESK_MODULE_v2\Backtest_tick_module\data\feature\2025-12-19\3060.parquet')
df.columns 


Index(['type', 'time', 'price', 'volume', 'tick_type', 'vwap', 'vwap_5m',
       'day_high', 'ratio_15s_180s', 'ratio_15s_180s_w321', 'ratio_30s_180s',
       'ratio_30s_180s_w321', 'ratio_60s_180s', 'ratio_60s_180s_w321',
       'ratio_buy5min_54321', 'ratio_buy5min_33221', 'ratio_buy5min',
       'ratio_buy3min_321', 'ratio_buy3min', 'high_1m', 'low_1m', 'high_2m',
       'low_2m', 'high_3m', 'low_3m', 'high_5m', 'low_5m', 'low_10m',
       'low_15m', 'ratio_10s_30s', 'ratio_15s_60s', 'ratio_30s_120s',
       'ratio_15s_45s', 'ratio_15s_30s', 'vol_buy_15s', 'cum_buy_vol',
       'cum_sell_vol', 'cum_total_vol', 'bid_volume_5level',
       'ask_volume_5level', 'bid_ask_ratio', 'obi', 'bid1_volume',
       'bid2_volume', 'bid3_volume', 'bid4_volume', 'bid5_volume',
       'ask1_volume', 'ask2_volume', 'ask3_volume', 'ask4_volume',
       'ask5_volume'],
      dtype='object')

In [8]:
import pandas as pd
import numpy as np

# 1. 讀取檔案
path1 = r'C:\Users\User\Documents\HFT\Holdwin-trade_1215\holdwin-1215\3060_ticks_20251219_193503.parquet'
path2 = r'C:\Users\User\Documents\HFT\BACK_TESK_MODULE_v2\Backtest_tick_module\data\feature\2025-12-19\3060.parquet'

df1 = pd.read_parquet(path1)
df2 = pd.read_parquet(path2)

# ==========================================
# 2. 資料前處理與欄位對齊
# ==========================================
rename_map = {
    'Datetime': 'time',
    'Type': 'type',
    'Price': 'price',
    'ask_bid_ratio': 'bid_ask_ratio' 
}

# 定義要比較的欄位
cols_to_check = [
    'price', 
    'tick_type', 
    'ratio_15s_180s', 
    'ratio_15s_180s_w321', 
    'bid_ask_ratio'
]

# 定義數值欄位 (用於填補 0.0)
numeric_columns = [
    'price', 
    'ratio_15s_180s', 
    'ratio_15s_180s_w321', 
    'bid_ask_ratio'
]

# 選取並改名
df1_clean = df1.rename(columns=rename_map)[['time', 'type'] + cols_to_check].copy()
df2_clean = df2[['time', 'type'] + cols_to_check].copy()

# ------------------------------------------
# 關鍵步驟：統一資料格式以利排序
# ------------------------------------------
def clean_and_fill(df):
    # 數值欄位填補 0.0
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
    
    # tick_type 填補 "0" 並轉字串
    if 'tick_type' in df.columns:
        # 先轉成 string 處理 None，再統一格式
        df['tick_type'] = df['tick_type'].fillna(0).astype(str)
        # 處理 '1.0' (str) vs '1' (str) 的問題：若能轉 float 則先轉 float 再轉 int
        # 這裡用一個簡單的 apply 來清洗
        def clean_str_int(x):
            try:
                return str(int(float(x)))
            except:
                return str(x)
        df['tick_type'] = df['tick_type'].apply(clean_str_int)
    return df

df1_clean = clean_and_fill(df1_clean)
df2_clean = clean_and_fill(df2_clean)

# ==========================================
# 3. 解決順序問題：排序 + 產生 Rank ID
# ==========================================
print("🔄 正在進行組內排序以忽略順序差異...")

# 排序邏輯：先依時間、類型，再依「價格」排序。
# 如果價格也一樣，我們再用 tick_type 輔助排序，確保兩邊順序一致。
sort_keys = ['time', 'type', 'price', 'tick_type']

df1_sorted = df1_clean.sort_values(by=sort_keys)
df2_sorted = df2_clean.sort_values(by=sort_keys)

# 產生流水號 (CumCount)：這會給同一秒同一類型的資料打上 0, 1, 2 標籤
# 這樣 35.05 (rank 0) 就會對到 35.05 (rank 0)，而 35.10 (rank 1) 對到 35.10 (rank 1)
df1_sorted['rank'] = df1_sorted.groupby(['time', 'type']).cumcount()
df2_sorted['rank'] = df2_sorted.groupby(['time', 'type']).cumcount()

# ==========================================
# 4. 合併資料 (加入 Rank 作為 Key)
# ==========================================
merged_df = pd.merge(
    df1_sorted, 
    df2_sorted, 
    on=['time', 'type', 'rank'], # <--- 加入 rank
    how='inner', 
    suffixes=('_df1', '_df2')
)

print(f"✅ 資料合併完成 (已校正順序)。共配對到 {len(merged_df)} 筆資料。\n")

# ==========================================
# 5. 逐一欄位比對
# ==========================================

difference_found = False

for col in cols_to_check:
    col_df1 = f"{col}_df1"
    col_df2 = f"{col}_df2"
    
    series1 = merged_df[col_df1]
    series2 = merged_df[col_df2]

    if col in numeric_columns:
        # 因為前面已經 fillna(0.0) 過了，這裡直接比較即可
        is_diff = ~np.isclose(series1, series2, rtol=1e-05, atol=1e-08)
    else:
        # 字串比對
        is_diff = series1 != series2

    diff_rows = merged_df[is_diff]
    
    if not diff_rows.empty:
        difference_found = True
        print(f"🔴 發現差異: 欄位 [{col}]")
        print(f"   共有 {len(diff_rows)} 筆不一致。前 5 筆差異如下：")
        print(diff_rows[['time', 'type', col_df1, col_df2]].head(5).to_string(index=False))
        print("-" * 60)
    else:
        print(f"✅ 檢查通過: 欄位 [{col}] 完全一致。")

if not difference_found:
    print("\n🎉 恭喜！排除 [NaN] 與 [順序差異] 後，所有資料皆一致！")

🔄 正在進行組內排序以忽略順序差異...
✅ 資料合併完成 (已校正順序)。共配對到 57933 筆資料。

✅ 檢查通過: 欄位 [price] 完全一致。
🔴 發現差異: 欄位 [tick_type]
   共有 1962 筆不一致。前 5 筆差異如下：
                      time  type tick_type_df1 tick_type_df2
2025-12-19 09:00:15.574286 Trade             0             1
2025-12-19 09:00:15.633366 Trade             0             1
2025-12-19 09:00:15.915979 Trade             1             2
2025-12-19 09:00:15.940033 Trade             0             1
2025-12-19 09:00:16.164766 Trade             0             2
------------------------------------------------------------
🔴 發現差異: 欄位 [ratio_15s_180s]
   共有 9620 筆不一致。前 5 筆差異如下：
                      time  type  ratio_15s_180s_df1  ratio_15s_180s_df2
2025-12-19 09:00:15.574237 Trade                12.0                 0.0
2025-12-19 09:00:15.574286 Trade                12.0                 0.0
2025-12-19 09:00:15.586111 Trade                12.0                 0.0
2025-12-19 09:00:15.587537 Trade                12.0                 0.0
2025-12-19 09:00:15.599

In [3]:
import pandas as pd

df =pd.read_parquet(r'C:\Users\User\Documents\HFT\BACK_TESK_MODULE_v2\5498_ticks_20251222_131605.parquet')
df.columns

Index(['Timestamp', 'Datetime', 'StockCode', 'Type', 'Price', 'Volume',
       'TotalVolume', 'tick_type', 'ratio_15s_180s', 'ratio_30s_180s',
       'ratio_10s_30s', 'ratio_15s_45s', 'ratio_15s_180s_w321',
       'ratio_30s_180s_w321', 'ratio_buy3min', 'ratio_buy3min_321',
       'amplitude_3min', 'ask_bid_ratio', 'Ask1_Price', 'Ask1_Volume',
       'Bid1_Price', 'Bid1_Volume'],
      dtype='object')